In [0]:
from pyspark.sql import functions as f, Window 
from pyspark.sql.types import *

data = [(101, 'Rahul', 'Bangalore', '2025-01-01'), (101, 'Rahul', 'Mysore', '2025-03-01'), (102, 'Amit', 'Pune', '2025-02-01'), (102, 'Amit', 'Mumbai', '2025-04-01'), (103, 'John', 'Chennai', '2025-01-15')]

schema = StructType(
    [
        StructField('customer_id',IntegerType(),False),
        StructField('name',StringType()),
        StructField('city',StringType()),
        StructField('updated_date',StringType())
    ]
)

customers = spark.createDataFrame(data=data,schema=schema)

w1 = Window.partitionBy('customer_id').orderBy(f.col('updated_date').desc())

customers = customers.withColumn('rn',f.dense_rank().over(w1))

customers = customers.filter(f.col('rn')==1).select('customer_id','name','city','updated_date')

display(customers)

customers.write.format('delta').mode('overwrite').option('overwriteSchema','True').saveAsTable('silver_customers')



In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

customers = customers.withColumn(
    "updated_date",
    F.to_date("updated_date")
)

window_spec = (
    Window.partitionBy("customer_id")
          .orderBy(F.col("updated_date").desc())
)

silver_customers = (
    customers
    .withColumn(
        "rn",
        F.row_number().over(window_spec)
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

(
    silver_customers.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_customers")
)